# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`

This notebook demonstrates how to load, explore, and perform basic analysis on the [FAIR^2 dataset](https://sen.science/doi/10.71728/senscience.y7m0-f273) using the [`mlcroissant`](https://pypi.org/project/mlcroissant/) library, following the Croissant data format and metadata schema.

### Dataset Source
This dataset describes ordered logistic regression results on predictors of adoption of indigenous/modern knowledge in rangeland management by pastoralist households in Northern Kenya.

> Croissant schema URL: [https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json](https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json)


In [ ]:
# Ensure `mlcroissant` is installed
!pip install -q mlcroissant

## 1. Data Loading
Load the dataset and its metadata from the Croissant schema URL.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load metadata using mlcroissant
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # This is an MLCroissant Metadata object

# Print general metadata information (convert to dict for pretty print)
meta_dict = metadata.to_json()
print(f"{meta_dict['name']}\n\n{meta_dict['description']}")

## 2. Data Overview
Let's examine the available **record sets** and their fields using their `@id` fields.

In [ ]:
# List available record sets and their fields by @id
record_sets = [rs for rs in dataset.record_sets]
if not record_sets:
    print('No record sets found in the dataset metadata.')
else:
    for rs in record_sets:
        print(f"Record Set Name: {rs.name}")
        print(f"  @id: {rs.id}")
        print(f"  Fields:")
        for field in rs.fields:
            print(f"    - {field.name} (@id: {field.id})")
        print()

## 3. Data Extraction
Load data for each available record set (by `@id`) into DataFrames for further analysis.

In [ ]:
# Collect record set @ids dynamically
record_set_ids = [rs.id for rs in dataset.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    # Records generator
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f'Record set "{record_set_id}" loaded with {len(df)} records.')
    # Show field names (column names) for the first record set
    if len(dataframes) == 1:
        first_record_set = record_set_id
        print(f"Columns for '{record_set_id}': {df.columns.tolist()}")
        display(df.head())

## 4. Exploratory Data Analysis (EDA)

We demonstrate filtering, normalization, and grouping for a numeric field using the first available record set as example. All field references are by `@id` as required.

In [ ]:
if not record_set_ids:
    print("No record sets to perform EDA on.")
else:
    # Select the first record set for demonstration
    record_set_id = first_record_set
    df = dataframes[record_set_id]

    print(f'Columns available in record set ({record_set_id}):')
    print(df.columns.tolist())

    # Attempt to auto-detect a numeric field (by pandas dtype)
    numeric_fields = [col for col in df.columns
                     if str(df[col].dtype) in ['int64', 'float64', 'Int64', 'Float64']]

    if numeric_fields:
        numeric_field = numeric_fields[0]
        print(f"Using numeric field for analysis: '{numeric_field}'")
    else:
        # Fallback: try to coerce all columns to numeric, pick first successful
        numeric_field = None
        for col in df.columns:
            try:
                coerced = pd.to_numeric(df[col], errors='coerce')
                if coerced.notnull().sum() > 0:
                    numeric_field = col
                    df[col] = coerced
                    print(f"Using '{col}' (converted to numeric) as numeric field.")
                    break
            except:
                continue
        if numeric_field is None:
            print("No numeric field found in the first record set. Skipping EDA.")

    if numeric_field is not None:
        # Basic filtering: choose threshold as the mean
        threshold = df[numeric_field].mean()
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records where '{numeric_field}' > {threshold:.2f}: {len(filtered_df)} rows")
        display(filtered_df.head())

        # Normalize
        normalized_col = f"{numeric_field}_normalized"
        filtered_df[normalized_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print("Normalized values (first 5):")
        display(filtered_df[[numeric_field, normalized_col]].head())

        # Attempt to group by the first non-numeric field (if such exists)
        group_fields = [col for col in df.columns if col != numeric_field and (df[col].dtype == 'object')]
        if group_fields:
            group_field = group_fields[0]
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
            print(f"Mean of '{numeric_field}' grouped by '{group_field}':")
            display(grouped_df.head())
        else:
            print("No suitable group-by field found for grouping.")

## 5. Visualization

Visualize the distribution of the selected numeric field (if found).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if not record_set_ids or numeric_field is None:
    print("No numeric field found for visualization.")
else:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), bins=20, kde=True)
    plt.title(f"Distribution of {numeric_field} (record set: {record_set_id})")
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

## 6. Conclusion

- This notebook demonstrated loading FAIR^2 dataset metadata and records with `mlcroissant` using Croissant schema URLs.
- We identified record sets and their fields by `@id`, extracted tabular data, performed basic filtering and normalization on a numeric field, and visualized value distribution.
- For actual research applications, you should consult the dataset's full schema and documentation for the meaning of each field and ensure ethical use of potentially sensitive data (e.g., gender, location).